# Notebook 2: Model development
Popularity baseline, content-based, item-kNN, SVD, hybrid, evaluation and model comparison.
Everything is imported from the `recommender` package so notebook and API use the same code.

**Protocol (fixed before we look at any numbers):**
1. Chronological leave-last-out: last rating = test, second-to-last = validation.
2. Choose model variants and hybrid weights on **validation** only.
3. Refit on train + validation and score the **test** item once.

In [ ]:
import os
from pathlib import Path

# Run from the project root so `recommender` and the data/ folder are found
if Path.cwd().name == "notebooks":
    os.chdir("..")

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from recommender.config import DATA_DIR, GENRE_COLS, MIN_USER_RATINGS
from recommender.data import load_movielens, validate_data

pd.set_option("display.width", 140)
print("Working directory:", Path.cwd())

import evaluate as ev
from recommender.collaborative_model import ItemKNN, SVDModel
from recommender.config import COMPONENTS
from recommender.content_model import ContentModel
from recommender.evaluation import (
    catalog_coverage, intra_list_diversity, mean_popularity, rank_of_target, ranking_metrics,
)
from recommender.inference import Recommender
from recommender.preprocessing import Index, build_matrix, temporal_split
from recommender.ranking import (
    PopularityModel, RandomModel, combine, normalize_components, top_k_indices,
)

## 1. Split the data the right way
A random split would let the model train on a user's *future* ratings and test on their
past, which leaks information. Instead we hold out each user's **most recent** rating.

In [ ]:
movies, ratings = load_movielens(DATA_DIR)
validate_data(movies, ratings)
splits = temporal_split(ratings, MIN_USER_RATINGS)
index = Index.build(splits.train_full.user_id, movies.movie_id)

print(f"train: {len(splits.train):,} | validation: {len(splits.val):,} | test: {len(splits.test):,}")
seen_val = build_matrix(splits.train, index, "binary").toarray() > 0
target_val = ev.targets_for(splits.val, index)
print("users:", index.n_users, "| movies:", index.n_items)

## 2. Baselines: random and popularity
Any real model must beat **random**; a personalised model should also beat **popularity**.

In [ ]:
popularity = PopularityModel().fit(splits.train, index)
top = np.argsort(-popularity.scores)[:10]
pd.DataFrame({"title": movies.set_index("movie_id").loc[index.movie_ids[top], "title"].values,
              "weighted_score": popularity.scores[top].round(3),
              "n_ratings": popularity.item_counts[top].astype(int)})

## 3. Content-based model
Movies are similar if their genre vectors are similar (cosine). A user's score for a movie
is its mean similarity to the movies they liked.

In [ ]:
content = ContentModel().fit(movies, index)
query = movies[movies.title.str.contains("Toy Story", na=False)]
demo_movie = int(query.movie_id.iloc[0]) if len(query) else int(movies.movie_id.iloc[0])
items, scores = content.similar_items(index.movie_to_idx[demo_movie], 10)
print("Query:", movies.set_index("movie_id").loc[demo_movie, "title"])
pd.DataFrame({"title": movies.set_index("movie_id").loc[index.movie_ids[items], "title"].values,
              "similarity": scores.round(3)})

**Observation:** many results have similarity exactly 1.0 because they share the same
genres. Genre-only content is coarse, so it is better as one signal inside the hybrid than alone.

## 4. Item-item collaborative filtering
Similarity comes from *how users rated* the movies, not from metadata. We try three rating
signals: `raw` (1-5), `binary` (did they rate it) and `centered` (rating minus the user's mean).
**Hypothesis to test, not assume:** mean-centering helps rating prediction, but our task is
ranking *which movie a user rates next*, where the implicit signal (`binary`) may compete.

In [ ]:
knn_variants = {s: ItemKNN(s).fit(splits.train, index) for s in ev.KNN_SIGNALS}
knn = knn_variants["centered"]
items, scores = knn.similar_items(index.movie_to_idx[demo_movie], 10)
print("Collaborative neighbours of:", movies.set_index("movie_id").loc[demo_movie, "title"])
pd.DataFrame({"title": movies.set_index("movie_id").loc[index.movie_ids[items], "title"].values,
              "similarity": scores.round(3)})

## 5. SVD (matrix factorisation)
Compress the user-item matrix into 20 latent factors and score items by reconstruction.

In [ ]:
svd_variants = {s: SVDModel(s).fit(splits.train, index) for s in ev.SVD_SIGNALS}
print("fitted SVD variants:", list(svd_variants))

## 6. Compare every variant on the validation set
Recall@10 equals HitRate@10 here because each user has exactly one relevant item.

In [ ]:
variants = {"random": RandomModel(), "popularity": popularity, "content": content}
variants.update({f"knn_{s}": m for s, m in knn_variants.items()})
variants.update({f"svd_{s}": m for s, m in svd_variants.items()})

val_scores = ev.score_models(variants, splits.train, index)
val_table = pd.DataFrame(
    {name: ev.evaluate_scores(s, seen_val, target_val) for name, s in val_scores.items()}
).T
val_table[["Recall@10", "MAP@10", "NDCG@10", "MRR"]].round(4).sort_values("NDCG@10", ascending=False)

In [ ]:
best_knn = max(ev.KNN_SIGNALS, key=lambda s: val_table.loc[f"knn_{s}", "NDCG@10"])
best_svd = max(ev.SVD_SIGNALS, key=lambda s: val_table.loc[f"svd_{s}", "NDCG@10"])
print("best kNN signal:", best_knn, "| best SVD signal:", best_svd)

**Write in your own words:** which signal won for each model, and why might that be?
Did mean-centering help or not on *this* task?

## 7. Hybrid: tune the blend on validation
Scores from different models live on different scales, so each is min-max normalised per
user (over unseen movies only) and then combined with weights that sum to 1. We search a grid
of weights and keep the best validation NDCG@10.

In [ ]:
components = {
    "knn": val_scores[f"knn_{best_knn}"],
    "svd": val_scores[f"svd_{best_svd}"],
    "content": val_scores["content"],
    "popularity": val_scores["popularity"],
}
normalized = normalize_components(components, seen_val)

rows = []
for w in ev.weight_grid(COMPONENTS, 0.1):
    final = combine(normalized, w, seen_val)
    ndcg = ranking_metrics(rank_of_target(final, seen_val, target_val), ks=(10,))["NDCG@10"]
    rows.append({**w, "val_NDCG@10": ndcg})
grid = pd.DataFrame(rows).sort_values("val_NDCG@10", ascending=False)
best_weights = {c: float(grid.iloc[0][c]) for c in COMPONENTS}
print("best weights:", best_weights)
grid.head(8).round(4)

## 8. Final test evaluation (score the test item once)
Refit on train + validation, use the settings chosen above, and add beyond-accuracy metrics.

In [ ]:
seen_test = build_matrix(splits.train_full, index, "binary").toarray() > 0
target_test = ev.targets_for(splits.test, index)
parts = ev.fit_components(splits.train_full, movies, index, best_knn, best_svd)
raw = ev.score_models({"random": RandomModel(), **parts}, splits.train_full, index)
norm = normalize_components({n: raw[n] for n in COMPONENTS}, seen_test)

finals = {
    "random": raw["random"], "popularity": raw["popularity"], "content": raw["content"],
    f"knn ({best_knn})": raw["knn"], f"svd ({best_svd})": raw["svd"],
    "HYBRID (tuned)": combine(norm, best_weights, seen_test),
}
for dropped in COMPONENTS:  # ablation
    rest = {n: w for n, w in best_weights.items() if n != dropped}
    if sum(rest.values()) > 0:
        rest = {n: w / sum(rest.values()) for n, w in rest.items()}
        finals[f"hybrid without {dropped}"] = combine(norm, rest, seen_test)

table = {}
for name, s in finals.items():
    m = ev.evaluate_scores(s, seen_test, target_test)
    top10 = top_k_indices(s, 10)
    m["Coverage@10"] = catalog_coverage(top10, index.n_items)
    m["AvgPopularity@10"] = mean_popularity(top10, parts["popularity"].item_counts)
    m["Diversity@10"] = intra_list_diversity(top10, parts["content"].similarity)
    table[name] = m
test_table = pd.DataFrame(table).T.round(4)
test_table[["Precision@10", "Recall@10", "MAP@10", "NDCG@10", "MRR", "Coverage@10", "Diversity@10"]]

In [ ]:
main = [r for r in test_table.index if not r.startswith("hybrid without")]
test_table.loc[main, ["Recall@10", "MAP@10", "NDCG@10"]].plot(kind="bar", figsize=(9, 4))
plt.title("Test-set comparison (K=10)"); plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.show()

**Read the ablation rows:** if removing a component barely changes the score, it is not
pulling its weight. Compare accuracy against coverage and diversity: the most accurate model is not
always the best product.

## 9. Using the trained system: personalised, cold start, similar movies
`Recommender.fit` trains on **all** ratings, which is what the API serves.

In [ ]:
rec = Recommender(knn_signal=best_knn, svd_signal=best_svd, weights=best_weights).fit(movies, ratings)
user = int(rec.index.user_ids[0])
print(f"Hybrid recommendations for user {user}")
display(rec.recommend_for_user(user, k=5)[["movie_id", "title", "score"]])
print("Cold start (unknown user, likes Drama)")
display(rec.recommend_for_user(999_999, k=5, preferred_genres=["Drama"])[["movie_id", "title", "score"]])
print("Similar movies (collaborative)")
display(rec.similar_movies(demo_movie, k=5, mode="collaborative")[["movie_id", "title", "score"]])

## 10. Conclusions (fill in from YOUR results)
1. Best single model and by how much it beats popularity: ...
2. Did the hybrid beat every single model? Which ablation mattered most: ...
3. Accuracy vs coverage/diversity trade-off: ...
4. Limitations of this evaluation (one held-out item per user, offline only): ...
5. What I would try next: ...